# The Hidden Hotel Price Game
## Why does the same room cost 3× more on Saturday — and can you actually beat the algorithm?

The trip finally made it out of the group chat. Flights booked, hotel sorted, 
everyone actually committed this time. And then someone checks the price again 
the next morning and it's gone up £40.

This is not bad luck. This is yield management — the science of charging exactly 
what the market will bear at any given moment. Hotels have been running this 
algorithm for decades. Most travellers have no idea it exists.

This analysis uses 119,390 real hotel bookings to reverse-engineer the pricing 
logic. What drives the rate up? When is the worst time to book? Can the system 
actually be predicted?

The data has answers. Some of them are inconvenient.

---
**Author:** Trupthi Raj  
**Tools:** Python · pandas · Plotly · Folium · scikit-learn  
**Dataset:** Hotel Booking Demand · 119,390 bookings · Portugal · 2015–2017  
**Key variable:** ADR (Average Daily Rate) — the price paid per room per night

In [44]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
import warnings
warnings.filterwarnings('ignore')

# ── Load data ──────────────────────────────────────────────
df = pd.read_csv('hotel_bookings.csv')

print("=" * 55)
print("DATASET LOADED")
print("=" * 55)
print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"\nHotel types:")
print(df['hotel'].value_counts())
print(f"\nReservation status:")
print(df['reservation_status'].value_counts())
print(f"\nADR (price) statistics:")
print(df['adr'].describe().round(2))
print(f"\nCancellation rate:")
rate = df['is_canceled'].mean() * 100
print(f"{rate:.1f}% of all bookings were cancelled")
print(f"\nYear range:")
print(df['arrival_date_year'].value_counts().sort_index())

DATASET LOADED
Rows:    119,390
Columns: 32

Hotel types:
hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64

Reservation status:
reservation_status
Check-Out    75166
Canceled     43017
No-Show       1207
Name: count, dtype: int64

ADR (price) statistics:
count    119390.00
mean        101.83
std          50.54
min          -6.38
25%          69.29
50%          94.58
75%         126.00
max        5400.00
Name: adr, dtype: float64

Cancellation rate:
37.0% of all bookings were cancelled

Year range:
arrival_date_year
2015    21996
2016    56707
2017    40687
Name: count, dtype: int64


## What We're Working With

119,390 bookings across two hotel types — City and Resort — spanning 2015 to 2017. 
The key variable is ADR (Average Daily Rate): the actual price paid per room per night.

A few things jump out immediately.

The average nightly rate is £101.83. The maximum is £5,400. Someone booked a hotel 
room for five thousand pounds a night and this dataset knows about it. We are removing 
that booking and not dwelling on it.

More importantly: **37% of all bookings were cancelled.** That is not a rounding error. 
More than one in three people who booked a room never showed up — and hotels price 
around this reality. It explains a lot of what we're about to find.

The data also contains a negative ADR of -£6.38. A hotel charging negative money is 
not a deal. It is a data error. It will be removed without ceremony.

In [45]:
df_clean = df.copy()

# Remove cancellations — we're analysing actual stays, not intentions
df_clean = df_clean[df_clean['is_canceled'] == 0]

# Remove negative ADR
df_clean = df_clean[df_clean['adr'] > 0]

# Remove extreme outliers (above 99.5th percentile)
adr_cap = df_clean['adr'].quantile(0.995)
df_clean = df_clean[df_clean['adr'] <= adr_cap]

# Fill missing children with 0
df_clean['children'] = df_clean['children'].fillna(0)

# Fill missing country with 'Unknown'
df_clean['country'] = df_clean['country'].fillna('Unknown')

# Remove zero-night stays
df_clean['total_nights'] = (df_clean['stays_in_weekend_nights'] +
                             df_clean['stays_in_week_nights'])
df_clean = df_clean[df_clean['total_nights'] > 0]

# Total revenue per booking
df_clean['total_revenue'] = df_clean['adr'] * df_clean['total_nights']

# Map month names to numbers
month_map = {'January':1,'February':2,'March':3,'April':4,
             'May':5,'June':6,'July':7,'August':8,
             'September':9,'October':10,'November':11,'December':12}
df_clean['arrival_month_num'] = df_clean['arrival_date_month'].map(month_map)

# Lead time buckets
df_clean['lead_time_bucket'] = pd.cut(
    df_clean['lead_time'],
    bins=[-1, 7, 30, 90, 180, 737],
    labels=['Last minute (0-7d)', 'Short (8-30d)',
            'Medium (31-90d)', 'Long (91-180d)',
            'Very early (180d+)']
)

# ── Audit trail ────────────────────────────────────────────
print("=" * 55)
print("CLEANING AUDIT")
print("=" * 55)
print(f"Original rows:               {len(df):,}")
print(f"After removing cancellations:{len(df[df['is_canceled']==0]):,}")
print(f"Negative ADR removed:        {(df['adr'] <= 0).sum()} rows")
print(f"ADR cap applied at:          £{adr_cap:.2f}")
print(f"Clean rows remaining:        {len(df_clean):,}")
print(f"\nADR after cleaning:")
print(df_clean['adr'].describe().round(2))
print(f"\nHotel split:")
print(df_clean['hotel'].value_counts())


df_clean.to_csv('hotel_clean.csv', index=False)
print(f"✅ Exported {len(df_clean):,} rows to hotel_clean.csv")

CLEANING AUDIT
Original rows:               119,390
After removing cancellations:75,166
Negative ADR removed:        1960 rows
ADR cap applied at:          £276.00
Clean rows remaining:        73,054

ADR after cleaning:
count    73054.00
mean       101.33
std         45.00
min          0.26
25%         69.70
50%         94.00
75%        125.10
max        276.00
Name: adr, dtype: float64

Hotel split:
hotel
City Hotel      45065
Resort Hotel    27989
Name: count, dtype: int64
✅ Exported 73,054 rows to hotel_clean.csv


##  What Got Cut Before the Real Talk Begins

Starting with **119,390 bookings**, here is what didn't make it to the final dataset:

- **43,017 cancellations removed** — a 37% cancellation rate. More than one in three 
  bookings never converted to an actual stay. Hotels price around this. We'll come back to it.
- **1,960 negative or zero ADR rows dropped** — not a deal, a data error.
- **Outliers capped at £276/night** — the £5,400 booking has been quietly escorted out.
- **Zero-night stays removed** — if no nights were slept, no price was paid.

**Clean rows remaining: 73,054** — confirmed stays, confirmed prices, real patterns.

| Metric | Value |
|--------|-------|
| Mean ADR | £101.33 |
| Median ADR | £94.00 |
| Std Dev | £45.00 |
| Range | £0.26 – £276.00 |
| City Hotel bookings | 45,065 |
| Resort Hotel bookings | 27,989 |

City hotels dominate the volume. Resort hotels, as we'll see, have their own 
pricing logic entirely.

In [46]:
import folium
import branca.colormap as cm

# Country centroids
country_coords = {
    'PRT': [39.3999, -8.2245], 'GBR': [55.3781, -3.4360],
    'FRA': [46.2276, 2.2137], 'ESP': [40.4637, -3.7492],
    'DEU': [51.1657, 10.4515], 'ITA': [41.8719, 12.5674],
    'IRL': [53.1424, -7.6921], 'BEL': [50.5039, 4.4699],
    'BRA': [-14.2350, -51.9253], 'NLD': [52.1326, 5.2913],
    'USA': [37.0902, -95.7129], 'CHE': [46.8182, 8.2275],
    'AUT': [47.5162, 14.5501], 'POL': [51.9194, 19.1451],
    'CHN': [35.8617, 104.1954], 'ROU': [45.9432, 24.9668],
    'NOR': [60.4720, 8.4689], 'SWE': [60.1282, 18.6435],
    'DNK': [56.2639, 9.5018], 'RUS': [61.5240, 105.3188],
    'AUS': [-25.2744, 133.7751], 'CAN': [56.1304, -106.3468],
    'AGO': [-11.2027, 17.8739], 'MOZ': [-18.6657, 35.5296],
    'ISR': [31.0461, 34.8516], 'ARG': [-38.4161, -63.6167],
    'MEX': [23.6345, -102.5528], 'ZAF': [-30.5595, 22.9375],
    'JPN': [36.2048, 138.2529], 'KOR': [35.9078, 127.7669],
    'SGP': [1.3521, 103.8198], 'HUN': [47.1625, 19.5033],
    'CZE': [49.8175, 15.4730], 'GRC': [39.0742, 21.8243],
    'TUR': [38.9637, 35.2433], 'MAR': [31.7917, -7.0926],
    'TUN': [33.8869, 9.5375], 'EGY': [26.8206, 30.8025],
    'NGA': [9.0820, 8.6753], 'KEN': [-0.0236, 37.9062]
}

# Aggregate by country
country_adr = df_clean.groupby('country').agg(
    avg_adr=('adr', 'mean'),
    bookings=('adr', 'count')
).reset_index()

# Filter noise
country_adr = country_adr[
    (country_adr['bookings'] >= 50) &
    (country_adr['country'] != 'Unknown')
]

# Colour scale
colormap = cm.LinearColormap(
    colors=['#9FE1CB', '#1D9E75', '#534AB7', '#D4537E'],
    vmin=country_adr['avg_adr'].min(),
    vmax=country_adr['avg_adr'].max(),
    caption='Average Daily Rate (£)'
)

# Build map
m = folium.Map(
    location=[20, 0],
    zoom_start=2,
    tiles='CartoDB dark_matter'
)

for _, row in country_adr.iterrows():
    code = row['country']
    if code in country_coords:
        lat, lon = country_coords[code]
        folium.CircleMarker(
            location=[lat, lon],
            radius=max(6, min(30, row['bookings'] / 400)),
            color=colormap(row['avg_adr']),
            fill=True,
            fill_color=colormap(row['avg_adr']),
            fill_opacity=0.8,
            weight=1.5,
            popup=folium.Popup(
                f"""
                <div style='font-family:sans-serif; font-size:13px;'>
                <b>{code}</b><br>
                Avg ADR: <b>£{row['avg_adr']:.0f}</b><br>
                Bookings: <b>{row['bookings']:,}</b>
                </div>
                """,
                max_width=200
            )
        ).add_to(m)

colormap.add_to(m)

# Save
m.save('chart1_global_adr_map.html')
print(" Map saved as chart1_global_adr_map.html")
print(f"Countries plotted: {country_adr[country_adr['country'].isin(country_coords)].shape[0]}")
print(f"\nTop 5 countries by avg ADR:")
print(country_adr.nlargest(5, 'avg_adr')[['country','avg_adr','bookings']].to_string(index=False))
print(f"\nTop 5 countries by booking volume:")
print(country_adr.nlargest(5, 'bookings')[['country','avg_adr','bookings']].to_string(index=False))

from IPython.display import IFrame
IFrame('chart1_global_adr_map.html', width=900, height=500)

 Map saved as chart1_global_adr_map.html
Countries plotted: 32

Top 5 countries by avg ADR:
country    avg_adr  bookings
    LUX 129.024457       175
    JPN 120.518284       169
    USA 118.254990      1573
    CHE 117.131222      1293
    IRN 117.031667        60

Top 5 countries by booking volume:
country    avg_adr  bookings
    PRT  94.986865     19563
    GBR  90.704412      9584
    FRA 105.774563      8401
    ESP 109.196737      6236
    DEU 101.820413      6024


## The Guests Who Pay the Most Aren't the Ones Booking the Most

Portugal, the UK, France, Spain and Germany dominate the booking volume — 
expected, given both hotels are physically located in Portugal and Europe 
books close to home.

But volume and rate don't travel together. Luxembourg, Japan and the USA 
pay the highest average nightly rates — guests arriving from further away 
tend to book better rooms, stay longer, and aren't hunting for the cheapest 
available option. Portugal, despite generating nearly 20,000 bookings, 
averages just £95 a night.

Bubble size = booking volume. Colour = average ADR. The big teal cluster over 
Europe is volume. The purple dots are where the money is.

The pattern is consistent across hospitality: proximity drives volume, 
distance drives spend.

In [47]:
fig = go.Figure()

colors = {'City Hotel': '#534AB7', 'Resort Hotel': '#1D9E75'}

for hotel in ['City Hotel', 'Resort Hotel']:
    fig.add_trace(go.Violin(
        y=df_clean[df_clean['hotel'] == hotel]['adr'],
        name=hotel,
        box_visible=True,
        meanline_visible=True,
        fillcolor=colors[hotel],
        opacity=0.75,
        line_color='rgba(255,255,255,0.6)',
        points='outliers',
        marker=dict(
            color=colors[hotel],
            size=2,
            opacity=0.3
        )
    ))

fig.update_layout(
    title=dict(
        text='Resort Hotels Charge More — But City Hotels Are the Safer Bet<br><sup>ADR distribution by hotel type · confirmed stays only · 2015–2017</sup>',
        font=dict(size=18, color='white'),
        x=0.5
    ),
    yaxis=dict(
        title='Average Daily Rate (£)',
        gridcolor='rgba(255,255,255,0.08)',
        color='white',
        zerolinecolor='rgba(255,255,255,0.1)'
    ),
    xaxis=dict(color='white'),
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white', family='Inter, sans-serif'),
    violingap=0.4,
    violinmode='group',
    showlegend=True,
    legend=dict(
        bgcolor='rgba(255,255,255,0.05)',
        bordercolor='rgba(255,255,255,0.1)',
        borderwidth=1,
        font=dict(color='white')
    ),
    height=560,
    margin=dict(t=100, b=60, l=60, r=40)
)

fig.write_html('chart2_adr_violin.html')
fig.write_image('chart2_adr_violin.png', scale=2)
fig.show()

print(" Saved as chart2_adr_violin.html and chart2_adr_violin.png")

 Saved as chart2_adr_violin.html and chart2_adr_violin.png


## Two Hotels, Two Completely Different Pricing Philosophies

City hotels are consistent. The bulk of bookings cluster tightly between 
£80 and £130 — predictable pricing for a predictable product. Business 
travellers, short stays, midweek demand. The rate doesn't move much because 
the demand doesn't either.

Resort hotels are a different story. The median drops to £73 — cheaper on 
a typical night — but the distribution is far wider. When demand peaks, 
resort prices climb well above city rates. The shape of that violin is 
seasonal pricing in action: low floors, high ceilings, everything driven 
by when you show up.

**The takeaway:** if predictability matters, book a city hotel. If you're 
flexible on timing, a resort hotel offers both the best and worst value in 
this dataset — depending entirely on when you go.

In [48]:
# Map day of week
df_clean['day_of_week'] = pd.to_datetime(
    df_clean['arrival_date_year'].astype(str) + '-' +
    df_clean['arrival_month_num'].astype(str) + '-' +
    df_clean['arrival_date_day_of_month'].astype(str)
).dt.day_name()

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

heatmap_data = df_clean.groupby(
    ['arrival_month_num', 'day_of_week']
)['adr'].mean().reset_index()

heatmap_pivot = heatmap_data.pivot(
    index='arrival_month_num',
    columns='day_of_week',
    values='adr'
)[day_order]

fig = go.Figure(data=go.Heatmap(
    z=heatmap_pivot.values,
    x=day_order,
    y=month_labels,
    colorscale=[
        [0.0, '#1a1a2e'],
        [0.2, '#534AB7'],
        [0.5, '#D4537E'],
        [0.8, '#BA7517'],
        [1.0, '#FFD93D']
    ],
    colorbar=dict(
        title=dict(text='Avg ADR (£)', font=dict(color='white')),
        tickfont=dict(color='white'),
        thickness=15
    ),
    hoverongaps=False,
    hovertemplate='%{x}<br>%{y}<br>Avg ADR: £%{z:.0f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='When You Arrive Matters as Much as Where You Stay<br><sup>Average daily rate by arrival month and day of week · confirmed stays · 2015–2017</sup>',
        font=dict(size=18, color='white'),
        x=0.5
    ),
    xaxis=dict(
        title='Day of Week',
        color='white',
        gridcolor='rgba(255,255,255,0.05)'
    ),
    yaxis=dict(
        title='Month',
        color='white',
        gridcolor='rgba(255,255,255,0.05)'
    ),
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white', family='Inter, sans-serif'),
    height=500,
    margin=dict(t=100, b=60, l=80, r=40)
)

fig.write_html('chart3_calendar_heatmap.html')
fig.show()

print(" Saved as chart3_calendar_heatmap.html")

 Saved as chart3_calendar_heatmap.html


## The Pricing Pattern Hotels Don't Advertise

The heatmap makes it obvious. August is expensive on every day of the week. 
January is cheap on every day of the week. That much was expected.

What's more useful is the within-month pattern. **Sunday arrivals 
consistently command higher rates** across almost every month — 
leisure travellers checking in for the week ahead, paying peak prices 
for the privilege. Midweek arrivals in shoulder months like March, 
April and November represent the clearest pricing gaps in the dataset.

The practical read: if the trip has any flexibility at all, a Wednesday 
arrival in October costs materially less than a Sunday arrival in August. 
The hotel is often the same room. The algorithm is not.

In [49]:
# Prepare data
scatter_data = df_clean.groupby(
    ['arrival_month_num', 'lead_time_bucket', 'hotel']
).agg(
    avg_adr=('adr', 'mean'),
    avg_lead=('lead_time', 'mean'),
    bookings=('adr', 'count')
).reset_index()

scatter_data['arrival_month_num'] = scatter_data['arrival_month_num'].astype(str)

month_names = {
    '1': 'January', '2': 'February', '3': 'March',
    '4': 'April', '5': 'May', '6': 'June',
    '7': 'July', '8': 'August', '9': 'September',
    '10': 'October', '11': 'November', '12': 'December'
}
scatter_data['month_name'] = scatter_data['arrival_month_num'].map(month_names)

fig = px.scatter(
    scatter_data,
    x='avg_lead',
    y='avg_adr',
    animation_frame='arrival_month_num',
    animation_group='lead_time_bucket',
    size='bookings',
    color='hotel',
    hover_name='lead_time_bucket',
    hover_data={
        'avg_adr': ':.0f',
        'avg_lead': ':.0f',
        'bookings': ':,',
        'arrival_month_num': False
    },
    color_discrete_map={
        'City Hotel': '#534AB7',
        'Resort Hotel': '#1D9E75'
    },
    size_max=60,
    range_x=[0, 220],
    range_y=[60, 160],
    labels={
        'avg_lead': 'Average Lead Time (days)',
        'avg_adr': 'Average Daily Rate (£)',
        'hotel': 'Hotel Type'
    }
)

fig.update_layout(
    title=dict(
        text='Booking Earlier Does Not Always Mean Paying Less<br><sup>Lead time vs ADR by hotel type · animated by month · bubble size = booking volume</sup>',
        font=dict(size=18, color='white'),
        x=0.5
    ),
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white', family='Inter, sans-serif'),
    xaxis=dict(
        color='white',
        gridcolor='rgba(255,255,255,0.08)',
        zerolinecolor='rgba(255,255,255,0.1)'
    ),
    yaxis=dict(
        color='white',
        gridcolor='rgba(255,255,255,0.08)',
        zerolinecolor='rgba(255,255,255,0.1)'
    ),
    legend=dict(
        bgcolor='rgba(255,255,255,0.05)',
        bordercolor='rgba(255,255,255,0.1)',
        borderwidth=1,
        font=dict(color='white')
    ),
    height=560,
    margin=dict(t=100, b=60, l=60, r=40),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        bgcolor='#534AB7',
        font=dict(color='white'),
        buttons=[
            dict(label='▶ Play',
                 method='animate',
                 args=[None, dict(frame=dict(duration=800, redraw=True),
                                 fromcurrent=True)]),
            dict(label='⏸ Pause',
                 method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                   mode='immediate')])
        ]
    )]
)

fig.write_html('chart4_animated_scatter.html')
fig.show()

print(" Saved as chart4_animated_scatter.html")

 Saved as chart4_animated_scatter.html


## Lead Time Is a Weaker Signal Than Hotels Want You to Think

The conventional wisdom: book early, pay less. The data is less convinced.

City hotel rates stay relatively stable regardless of how far in advance 
the booking is made — last-minute and early bookers pay similar rates. 
Resort hotels show more variation, but the pattern shifts month to month. 
**In peak summer months, early bookers don't get a meaningful discount** — 
demand is high enough that the hotel doesn't need to incentivise early 
commitment.

Where lead time does matter is in shoulder months. Book a March or November 
stay more than 90 days out and the rate drops noticeably — hotels filling 
rooms in quieter periods are more willing to reward early commitment. 
The algorithm is not uniform. It is seasonal.

In [50]:
# Prepare data
treemap_data = df_clean.groupby(['country', 'hotel']).agg(
    total_revenue=('total_revenue', 'sum'),
    avg_adr=('adr', 'mean'),
    bookings=('adr', 'count')
).reset_index()

# Keep top countries by revenue for readability
top_countries = treemap_data.groupby('country')['total_revenue'].sum()
top_countries = top_countries.nlargest(20).index
treemap_data = treemap_data[treemap_data['country'].isin(top_countries)]

# Add a root level column
treemap_data['world'] = 'All Bookings'

fig = px.treemap(
    treemap_data,
    path=['world', 'country', 'hotel'],
    values='total_revenue',
    color='avg_adr',
    hover_data={
        'bookings': ':,',
        'avg_adr': ':.0f',
        'total_revenue': ':,.0f'
    },
    color_continuous_scale=[
        '#1a1a2e', '#534AB7', '#D4537E', '#BA7517', '#FFD93D'
    ],
    color_continuous_midpoint=treemap_data['avg_adr'].median()
)

fig.update_traces(
    textfont=dict(color='white', size=13),
    marker=dict(
        line=dict(width=1.5, color='#1a1a2e')
    ),
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Total Revenue: £%{customdata[2]:,.0f}<br>'
        'Avg ADR: £%{customdata[1]:.0f}<br>'
        'Bookings: %{customdata[0]:,}<extra></extra>'
    )
)

fig.update_layout(
    title=dict(
        text='Portugal Generates the Volume — But Not the Premium Rate<br><sup>Total revenue by country and hotel type · top 20 markets · confirmed stays · 2015–2017</sup>',
        font=dict(size=18, color='white'),
        x=0.5
    ),
    paper_bgcolor='#1a1a2e',
    font=dict(color='white', family='Inter, sans-serif'),
    coloraxis_colorbar=dict(
        title=dict(text='Avg ADR (£)', font=dict(color='white')),
        tickfont=dict(color='white'),
        thickness=15
    ),
    height=580,
    margin=dict(t=100, b=40, l=40, r=40)
)

fig.write_html('chart5_treemap_revenue.html')
fig.show()

print("Saved as chart5_treemap_revenue.html")

Saved as chart5_treemap_revenue.html


## The Countries That Book the Most Are Not the Countries That Spend the Most

Portugal accounts for the largest share of bookings by a significant margin. 
It also pays some of the lowest average rates in the dataset. High familiarity 
with the local market, proximity, and repeat booking behaviour all compress 
the price.

The more interesting signal is in the smaller boxes. **Switzerland, the USA 
and China generate far less volume but pay materially higher rates** — 
averaging above £115/night against Portugal's £95. These are guests arriving 
with different expectations, booking different room types, and providing 
disproportionate revenue relative to their booking share.

For a hotel's commercial team, the implication is straightforward: volume 
markets keep occupancy high, but premium markets protect the rate. 
Both matter — but they need to be managed differently.

In [51]:
# Calculate baseline and factor impacts
baseline = df_clean['adr'].mean()

# Factor impacts vs baseline
factors = {
    'Baseline ADR': baseline,
    'Resort Hotel': df_clean[df_clean['hotel'] == 'Resort Hotel']['adr'].mean() - baseline,
    'Summer Arrival\n(Jun–Aug)': df_clean[df_clean['arrival_month_num'].isin([6,7,8])]['adr'].mean() - baseline,
    'Weekend Arrival\n(Fri–Sun)': df_clean[df_clean['day_of_week'].isin(['Friday','Saturday','Sunday'])]['adr'].mean() - baseline,
    'Long Lead Time\n(90+ days)': df_clean[df_clean['lead_time'] >= 90]['adr'].mean() - baseline,
    'Last Minute\n(0–7 days)': df_clean[df_clean['lead_time'] <= 7]['adr'].mean() - baseline,
    'Winter Arrival\n(Dec–Feb)': df_clean[df_clean['arrival_month_num'].isin([12,1,2])]['adr'].mean() - baseline,
    'City Hotel': df_clean[df_clean['hotel'] == 'City Hotel']['adr'].mean() - baseline,
}

labels = list(factors.keys())
values = list(factors.values())

# Build waterfall colours
colors = []
for i, (label, val) in enumerate(factors.items()):
    if i == 0:
        colors.append('#534AB7')  # baseline
    elif val >= 0:
        colors.append('#1D9E75')  # positive
    else:
        colors.append('#D4537E')  # negative

# Measure types
measure = ['absolute'] + ['relative'] * (len(values) - 1)

fig = go.Figure(go.Waterfall(
    name='ADR Impact',
    orientation='v',
    measure=measure,
    x=labels,
    y=values,
    text=[f'£{v:+.1f}' if i > 0 else f'£{v:.1f}' for i, v in enumerate(values)],
    textposition='outside',
    textfont=dict(color='white', size=12),
    connector=dict(
        line=dict(color='rgba(255,255,255,0.2)', width=1, dash='dot')
    ),
    increasing=dict(marker=dict(color='#1D9E75')),
    decreasing=dict(marker=dict(color='#D4537E')),
    totals=dict(marker=dict(color='#534AB7'))
))

fig.update_layout(
    title=dict(
        text='What Actually Moves the Nightly Rate — and By How Much<br><sup>ADR impact of key booking factors vs dataset baseline of £101 · confirmed stays · 2015–2017</sup>',
        font=dict(size=18, color='white'),
        x=0.5
    ),
    yaxis=dict(
        title='Impact on ADR (£)',
        color='white',
        gridcolor='rgba(255,255,255,0.08)',
        zerolinecolor='rgba(255,255,255,0.3)'
    ),
    xaxis=dict(
        color='white',
        tickfont=dict(size=11)
    ),
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white', family='Inter, sans-serif'),
    showlegend=False,
    height=560,
    margin=dict(t=100, b=80, l=60, r=40)
)

fig.write_html('chart6_waterfall_adr.html')
fig.show()

print("✅ Saved as chart6_waterfall_adr.html")
print(f"\nBaseline ADR: £{baseline:.2f}")
print("\nFactor impacts:")
for label, val in list(factors.items())[1:]:
    print(f"  {label.replace(chr(10), ' ')}: {val:+.2f}")

✅ Saved as chart6_waterfall_adr.html

Baseline ADR: £101.33

Factor impacts:
  Resort Hotel: -10.59
  Summer Arrival (Jun–Aug): +28.66
  Weekend Arrival (Fri–Sun): +1.72
  Long Lead Time (90+ days): +1.19
  Last Minute (0–7 days): -7.23
  Winter Arrival (Dec–Feb): -26.72
  City Hotel: +6.57


## The Factors That Move the Price — Ranked by Actual Impact

Starting from a baseline of £101/night, this is what the data says 
actually moves the rate:

**Season is the dominant factor — by a wide margin.** A summer arrival 
adds £29 to the nightly rate on average. A winter arrival removes £27. 
No other variable comes close. If the trip has any seasonal flexibility, 
that decision alone is worth more than any other booking choice combined.

Everything else is secondary. Weekend arrivals add less than £2. Booking 
90+ days out adds just over £1. Last-minute bookings save around £7 — 
real, but not the windfall the internet suggests.

**The practical hierarchy:** pick the right month first. Then pick the 
hotel type. Day of week and lead time are marginal adjustments on top 
of a decision that's already been made.

## What 73,054 Bookings Actually Teach You About Hotel Pricing

The trip made it out of the group chat. The algorithm was always running 
in the background. Here is what six charts and 73,054 confirmed stays 
revealed about how hotels actually set prices — and what to do about it.

---

### The Findings

**1. Season is the only variable that materially moves the rate.**
A summer arrival costs £29 more per night than the baseline. A winter 
arrival saves £27. No other factor — not lead time, not day of week, 
not hotel type — comes close. The advice to "book early to save money" 
is technically true and almost entirely irrelevant if you're arriving 
in August.

**2. Resort and city hotels are not competing on the same terms.**
City hotels are consistent — predictable pricing for a predictable 
product. Resort hotels have lower median rates but dramatic seasonal 
swings. They offer both the best and worst value in this dataset, 
depending entirely on when you show up.

**3. The 37% cancellation rate shapes everything.**
More than one in three bookings was cancelled. Hotels price with this 
in mind — rates are set to account for no-shows, which is partly why 
last-minute availability exists at all. The algorithm is not just 
responding to demand. It is hedging against it.

**4. Distance correlates with spend.**
Portugal generates the most bookings. It also pays the lowest rates. 
Guests from Luxembourg, Japan and the USA arrive less frequently — 
and spend significantly more when they do. Proximity drives volume. 
Distance drives revenue.

**5. Lead time is a weaker lever than advertised.**
Booking 90+ days out adds just over £1 to the average nightly rate. 
The myth of the early-bird discount is not well supported here — at 
least not in peak season, when hotels have no incentive to reward 
patience.

---

### What a Hotel Commercial Team Would Do With This

- **Protect summer rates aggressively.** Demand justifies it and guests 
  will pay. The data confirms it.
- **Use winter pricing to drive occupancy**, not revenue. The floor is 
  already low — the goal is filling rooms, not defending the rate.
- **Target international high-spend markets** for premium room upsells. 
  They arrive with higher willingness to pay and book less on price alone.
- **Take the 37% cancellation rate seriously** as a revenue management 
  problem, not just an operational one. It is baked into every price 
  on the rate card.

---

### More Projects Like This

If this analysis was useful, the rest of the portfolio lives here:  
🔗 [github.com/trupthiraj](https://github.com/trupthiraj)